<a href="https://colab.research.google.com/github/AnasSalah555/zotero-citation-cleaner/blob/main/repair_duplicated_ref_using_zotero%26colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import zipfile, json, re, html
from google.colab import files

uploaded = files.upload()
input_docx = list(uploaded.keys())[0]
output_docx = "Manuscript_Deduplicated_Clean.docx"

with zipfile.ZipFile(input_docx, 'r') as zin:
    xml_data = zin.read('word/document.xml').decode('utf-8')
    all_files = {name: zin.read(name) for name in zin.namelist()}

merge_pat = re.compile(r'</w:instrText></w:r><w:r(?:\s[^>]*)?>(?:<w:rPr>(?:(?!</w:rPr>).)*</w:rPr>)?<w:instrText(?:\s[^>]*)?>')
while merge_pat.search(xml_data):
    xml_data = merge_pat.sub('', xml_data)

canonical_by_doi = {}
canonical_by_title = {}
pattern = re.compile(r'(ADDIN ZOTERO_ITEM CSL_CITATION\s*)(\{.*?\})(\s*</w:instrText>)', re.DOTALL)

def unify_citations(match):
    prefix, json_str, suffix = match.group(1), match.group(2), match.group(3)
    try:
        data = json.loads(html.unescape(json_str))
    except Exception:
        return match.group(0)

    new_items = []
    seen_in_bracket = set()
    for item in data.get('citationItems', []):
        idata = item.get('itemData', {})
        doi = idata.get('DOI', '').strip().lower()
        title = re.sub(r'[^a-z0-9]', '', idata.get('title', '').lower())

        canon = None
        if doi and doi in canonical_by_doi:
            canon = canonical_by_doi[doi]
        elif title and title in canonical_by_title:
            canon = canonical_by_title[title]

        if canon is not None:
            item['id'] = canon['id']
            item['uris'] = canon['uris']
            item['itemData'] = canon['itemData']
        else:
            canon = {'id': item.get('id'), 'uris': item.get('uris'), 'itemData': idata}
            if doi: canonical_by_doi[doi] = canon
            if title: canonical_by_title[title] = canon

        uri_key = tuple(canon['uris'])
        if uri_key not in seen_in_bracket:
            seen_in_bracket.add(uri_key)
            new_items.append(item)

    data['citationItems'] = new_items
    out_json = json.dumps(data, ensure_ascii=False)
    out_json_xml = out_json.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
    return prefix + out_json_xml + suffix

xml_cleaned = pattern.sub(unify_citations, xml_data)
all_files['word/document.xml'] = xml_cleaned.encode('utf-8')

with zipfile.ZipFile(output_docx, 'w', zipfile.ZIP_DEFLATED) as zout:
    for name, content in all_files.items():
        zout.writestr(name, content)

import IPython.display
IPython.display.clear_output(wait=True)
print("Processing completed successfully. Downloading clean file...")
files.download(output_docx)

Processing completed successfully. Downloading clean file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

hello scientests!! #
==============================================================================
# Ownership & Contact Information
 ==============================================================================
# This notebook and its source code are proudly owned and developed by Anas EL-Ghandour.
# If you encounter any issues, bugs, or have inquiries, please feel free to
# contact me at: anaselghandour763@gmail.com
# ==============================================================================  
#Zotero Citation Duplication Cleaner & Unifier
# DESCRIPTION:
# This script automatically removes hidden Zotero/Mendeley duplicate citations
# from Word (.docx) documents by matching DOIs or titles, unifying their hidden
# URIs and IDs, and cleaning the XML structure without breaking the document.
#
# INSTRUCTIONS FOR USERS:
# 1. Run this code cell (Click the Play button ▶️).
# 2. When the "Choose Files" button appears, select your Word (.docx) file.
# 3. Wait a few seconds for the script to process the document XML.
# 4. The cleaned file ("Manuscript_Deduplicated_Clean.docx") will download
#    automatically to your computer.
# 5. Open the downloaded file in Microsoft Word (ensure Zotero is open),
#    and click "Refresh" on the Zotero ribbon to update all citations.
# ==============================================================================
